In [2]:
# hide
import numpy as np
import pyquist as pq
from pyquist.helper import pitch_name_to_pitch


def resample(x: pq.Audio, ratio: float) -> pq.Audio:
    """Read x at `ratio` times its original speed, with linear interpolation."""
    samples = np.asarray(x.samples)
    N = len(samples)
    M = int(N / ratio)                        # the duration scales by 1 / ratio
    p = np.arange(M) * ratio                  # fractional read positions in x
    p_floor = np.floor(p).astype(int)
    alpha = (p - p_floor)[:, np.newaxis]
    p_next = np.minimum(p_floor + 1, N - 1)
    y = (1 - alpha) * samples[p_floor] + alpha * samples[p_next]
    return pq.Audio(y, x.sample_rate)


def repitch(x: pq.Audio, k: float) -> pq.Audio:
    """Shift the pitch of x by k steps."""
    return resample(x, 2 ** (k / 12))

In [3]:
LOOP_START, LOOP_END = 0.5, 0.72095   # seconds: 206 periods of A#5, about 9744 audio samples
RELEASE = 3.84968                     # seconds: where the release begins


def clarinet(pitch: str, duration: float, loudness: float) -> pq.Audio:
    """A sampled clarinet with independent control over pitch, duration, and loudness."""
    recording = pq.Audio.from_file("../assets/clarinet.wav")
    k = pitch_name_to_pitch(pitch) - pitch_name_to_pitch("A#5")
    x = np.asarray(repitch(recording, k).samples)                          # pitch
    to_samples = recording.sample_rate / 2 ** (k / 12)
    s, e, r = (round(t * to_samples) for t in (LOOP_START, LOOP_END, RELEASE))
    n = max(0, round((duration * recording.sample_rate - s - (len(x) - r)) / (e - s)))
    x = np.concatenate([x[:s]] + [x[s:e]] * n + [x[r:]])                   # duration
    gain = 0.0 if loudness == 0 else 10 ** (80 * (loudness - 1) / 20)      # loudness
    return pq.Audio(gain * x, recording.sample_rate)


# A crescendo: four notes, each louder than the last
notes = [("D5", 0.7), ("F5", 0.8), ("A#5", 0.9), ("D6", 1.0)]
melody = pq.Score([(2.0 * i, {"pitch": pitch, "duration": 2.5, "loudness": loudness})
                   for i, (pitch, loudness) in enumerate(notes)])
pq.play(melody.render(clarinet), normalize=True)